In [72]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from implicit.als import AlternatingLeastSquares
import implicit
import os
import scipy.sparse as sparse

In [30]:
active_subjects = pd.read_csv(os.path.join("..", "..","data", "active_subjects.csv"))
audiences = pd.read_csv(os.path.join("..", "..","data", 'audiencies.csv'))
institutions = pd.read_csv(os.path.join("..", "..","data", 'institutions.csv'))
passive_subjects = pd.read_csv(os.path.join("..", "..","data", 'passive_subjects.csv'))

In [60]:
active_subjects.columns

Index(['audiencia_id', 'institucion_id', 'sujeto_pasivo_id', 'anio',
       'Nombre completo', 'Calidad', 'Trabaja para', 'Representa a'],
      dtype='object')

In [61]:
active_subjects.head()

,audiencia_id,institucion_id,sujeto_pasivo_id,anio,Nombre completo,Calidad,Trabaja para,Representa a
0,691146,AB023,634851,2024,Leslie Zapata,Gestor de intereses,NaN,Leslie Alejandra Zapata Vásquez
1,691322,AB023,634851,2024,Flora Flores,Gestor de intereses,NaN,Vecinos de Caquena
2,691521,AB023,634851,2024,Clara Blanco,Gestor de intereses,NaN,Clara Blanco Mamani
3,692878,AB023,634851,2024,Conrado Blanco,Gestor de intereses,NaN,Asoc. de Ganaderos de Guallatire
4,740813,AB023,634851,2024,Yessica Sanches,Gestor de intereses,NaN,Consejo ADI


In [62]:
active_id = (
    active_subjects[['Nombre completo', 'Calidad', 'Representa a']]
    .drop_duplicates(subset=['Nombre completo'])
    .reset_index(drop=True)
)

In [68]:
active_id.head()

,Nombre completo,Calidad,Representa a
0,Leslie Zapata,Gestor de intereses,Leslie Alejandra Zapata Vásquez
1,Flora Flores,Gestor de intereses,Vecinos de Caquena
2,Clara Blanco,Gestor de intereses,Clara Blanco Mamani
3,Conrado Blanco,Gestor de intereses,Asoc. de Ganaderos de Guallatire
4,Yessica Sanches,Gestor de intereses,Consejo ADI


In [66]:
active_id.shape

(376788, 3)

In [48]:
df = active_subjects[["sujeto_pasivo_id", "Nombre completo"]].copy()

In [49]:
df["count"] = 1

In [77]:
df_train = df.groupby('sujeto_pasivo_id').apply(lambda x: x.sample(frac=0.8, random_state=42)).reset_index(drop=True)
df_test = df.drop(df_train.index)

/var/folders/df/h6wb5c_53z92tgmtk94_mlfw0000gn/T/ipykernel_24671/1171520814.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train = df.groupby('sujeto_pasivo_id').apply(lambda x: x.sample(frac=0.8, random_state=42)).reset_index(drop=True)


In [78]:
df_train.head()

,sujeto_pasivo_id,Nombre completo,count
0,15,Gonzalo Velásquez,1
1,15,Jimena Norambuena,1
2,15,Simón Ortega Arriagada,1
3,15,Andres Rivera,1
4,15,Aida Alejandra Mena Olivares,1


In [79]:
user_items_test = {}

for row in df_test.itertuples():
    if row[1] not in user_items_test:
        user_items_test[row[1]] = []

    user_items_test[row[1]].append(row[2])

In [80]:
# Definicion de métricas (No editar)

def precision_at_k(r, k):
    assert 1 <= k <= r.size
    return (np.asarray(r)[:k]).mean()

def average_precision_at_k(r, k):
    r = np.asarray(r)
    n_rel = r.sum() # Número de items relevantes
    if n_rel == 0:
        return 0.
    vectorized_precision = np.vectorize(lambda i: precision_at_k(r, i))
    indices = np.arange(1, len(r) + 1)
    precisions = vectorized_precision(indices) # Precision@k por cada posición del ranking
    score = np.sum(precisions * r)
    return score / min(k, n_rel)

def dcg_at_k(r, k):
    r = np.asarray(r)[:k]
    if r.size:
        return np.sum(np.subtract(np.power(2, r), 1) / np.log2(np.arange(2, r.size + 2)))
    return 0.


def ndcg_at_k(r, k):
    idcg = dcg_at_k(sorted(r, reverse=True), k)

    if not idcg:
        return 0.
    return dcg_at_k(r, k) / idcg

In [81]:
user_items = {}
itemset = set()

for row in df_train.itertuples():
    if row[1] not in user_items:
        user_items[row[1]] = []

    user_items[row[1]].append(row[2])
    itemset.add(row[2])

itemset = np.sort(list(itemset))

sparse_matrix = np.zeros((len(user_items), len(itemset)))

for i, items in enumerate(user_items.values()):
    sparse_matrix[i] = np.isin(itemset, items, assume_unique=True).astype(int)

matrix = sparse.csr_matrix(sparse_matrix.T)

user_item_matrix = matrix.T.tocsr()

In [82]:
# Mapeo de user id a fila de la matriz sparse
user2row = {user_id: matrix_row for matrix_row, user_id in enumerate(user_items.keys())}
row2user = {matrix_row: user_id for user_id, matrix_row in user2row.items()}

# Mapeo de item id a columna de la matriz sparse
item2col = {item_id: matrix_col for matrix_col, item_id in enumerate(itemset)}
col2item = {matrix_col: item_id for item_id, matrix_col in item2col.items()}

In [83]:
def evaluate_model(model, n):
  mean_ap = 0. # o MAP
  mean_ndcg = 0.
  for user_id in user_items_test.keys():
    user_row = user2row[user_id]
    rec = model.recommend(user_row, user_item_matrix[user_row], n)[0]
    rec = [col2item[col] for col in rec]
    rel_vector = np.isin(rec, user_items_test[user_id], assume_unique=True).astype(int)
    mean_ap += average_precision_at_k(rel_vector, n)
    mean_ndcg += ndcg_at_k(rel_vector, n)

  mean_ap /= len(user_items_test)
  mean_ndcg /= len(user_items_test)

  return mean_ap, mean_ndcg

In [84]:
def show_recommendations(model, user, n):
  recommendations = model.recommend(userid=user, user_items=user_item_matrix[user], N=n)[0]
  return active_id.loc[recommendations]['Nombre completo']

In [85]:
def show_similar_movies(model, item, n=10):
  sim_items = model.similar_items(item, n)[0]
  return active_id.loc[sim_items]['Nombre completo']

In [86]:
model_als = implicit.als.AlternatingLeastSquares(factors=100, iterations=10, use_gpu=False)
model_als.fit(user_item_matrix)

100%|██████████| 10/10 [00:16<00:00,  1.66s/it]


In [87]:
show_recommendations(model_als, user=77, n=10)

161919      YOLANDA DEL CARMEN MORA LUNA
16787                      FELIPE Sierra
219930                   MIGUEL FOITZICK
248426                     Diego Canales
213019     Fernando German Pastene Rojas
77870                      Franco Burgos
113781             Cristopher Karamanoff
175695             Jose Antonio Troncoso
55663     Valeria Rocio Figueroa Herrera
143439                      Oscar Encina
Name: Nombre completo, dtype: object

In [88]:
maprec, ndcg = evaluate_model(model_als, n=10)
print('map: {}\nndcg: {}'.format(maprec, ndcg))

map: 0.04820276166336675
ndcg: 0.06494777353833124


In [89]:
for f in [50, 100, 200]:
    for a in [10, 40, 80]:
        for r in [0.01, 0.1, 0.5]:
            model = AlternatingLeastSquares(factors=f, alpha=a, regularization=r, iterations=20)
            model.fit(user_item_matrix)
            maprec, ndcg = evaluate_model(model, n=10)
            print(f"f={f}, a={a}, r={r}, MAP={maprec:.4f}, nDCG={ndcg:.4f}")


100%|██████████| 20/20 [00:10<00:00,  1.86it/s]


f=50, a=10, r=0.01, MAP=0.0696, nDCG=0.0950


100%|██████████| 20/20 [00:10<00:00,  1.86it/s]


f=50, a=10, r=0.1, MAP=0.0667, nDCG=0.0936


100%|██████████| 20/20 [00:10<00:00,  1.86it/s]


...

f=200, a=80, r=0.01, MAP=0.1470, nDCG=0.1944


100%|██████████| 20/20 [02:06<00:00,  6.35s/it]


f=200, a=80, r=0.1, MAP=0.1476, nDCG=0.1948


100%|██████████| 20/20 [02:08<00:00,  6.42s/it]


f=200, a=80, r=0.5, MAP=0.1464, nDCG=0.1929


Mejor resultado 
f=200, α≈80, regularization≈0.1, 
con:
MAP = 0.1476
nDCG = 0.1948